In [1]:
!gcloud auth application-default login

Your browser has been opened to visit:

    https://accounts.google.com/o/oauth2/auth?response_type=code&client_id=764086051850-6qr4p6gpi6hn506pt8ejuq83di341hur.apps.googleusercontent.com&redirect_uri=http%3A%2F%2Flocalhost%3A8085%2F&scope=openid+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fuserinfo.email+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fcloud-platform+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fsqlservice.login&state=TfpAzYcoE5PDyJCJ9gc1duqH1Hn0yj&access_type=offline&code_challenge=Us0BiKsGMv0vNr88lnjK7RQfrkPUFPVGxqWots-RZ9M&code_challenge_method=S256


Credentials saved to file: [/Users/yt4/.config/gcloud/application_default_credentials.json]

These credentials will be used by any library that requests Application Default Credentials (ADC).

Quota project "open-targets-genetics-dev" was added to ADC which can be used by Google client libraries for billing and quota. Note that some services may still bill the project owning the resource.


Updates are available for some Google Clo

In [2]:
import os

import hail as hl
import numpy as np
import pyspark.sql.functions as f
from pyspark.sql import DataFrame

from gentropy.common.session import Session
from gentropy.dataset.study_index import StudyIndex
from gentropy.dataset.summary_statistics import SummaryStatistics
from gentropy.dataset.study_locus import StudyLocus
from gentropy.susie_finemapper import SusieFineMapperStep
from gentropy.method.drug_enrichment_from_evid import chemblDrugEnrichment

"""Common utilities for the project."""

import os
from pathlib import Path
from gentropy.common.session import Session
import logging


def get_gcs_credentials() -> str:
    """Get the credentials for google cloud storage."""
    app_default_credentials = os.path.join(
        os.getenv("HOME", "."), ".config/gcloud/application_default_credentials.json"
    )

    service_account_credentials = os.path.join(
        os.getenv("HOME", "."), ".config/gcloud/service_account_credentials.json"
    )

    if Path(app_default_credentials).exists():
        return app_default_credentials
    else:
        raise FileNotFoundError("No GCS credentials found.")


def get_gcs_hadoop_connector_jar() -> str:
    """Get the google cloud storage hadoop connector for spark.

    This function will return the url to download the hadoop jar.
    """

    return (
        "https://storage.googleapis.com/hadoop-lib/gcs/gcs-connector-hadoop3-latest.jar"
    )


def gcs_conf(
    credentials_path=None, project="open-targets-genetics-dev"
) -> dict[str, str]:
    """Get the spark configuration with hadoop connector for google cloud storage."""
    credentials_path = credentials_path or get_gcs_credentials()
    return {
        "spark.driver.memory": "12g",
        "spark.kryoserializer.buffer.max": "500m",
        "spark.driver.maxResultSize": "2g",
        "spark.hadoop.fs.gs.impl": "com.google.cloud.hadoop.fs.gcs.GoogleHadoopFileSystem",
        "spark.jars": get_gcs_hadoop_connector_jar(),
        "spark.hadoop.google.cloud.auth.service.account.enable": "true",
        "spark.hadoop.fs.gs.project.id": project,
        "spark.hadoop.google.cloud.auth.service.account.json.keyfile": credentials_path,
        "spark.hadoop.fs.gs.requester.pays.mode": "AUTO",
    }


class GentropySession(Session):
    def __init__(self, *args, **kwargs):
        if "extended_spark_conf" in kwargs:
            kwargs["extended_spark_conf"].update(gcs_conf())
        else:
            kwargs["extended_spark_conf"] = gcs_conf()
        super().__init__(*args, **kwargs)

    @property
    def conf(self):
        logging.warning(
            "To change the config restart the session and use the `extended_spark_conf` parameter."
        )
        return self.spark.sparkContext.getConf().getAll()


session = GentropySession()


Loading BokehJS ...

/Users/yt4/Projects/gentropy/.venv/lib/python3.11/site-packages/pyspark/sql/pandas/functions.py:407: UserWarning:

In Python 3.6+ and Spark 3.0+, it is preferred to specify type hints for pandas UDF instead of specifying pandas UDF type which will be deprecated in the future releases. See SPARK-28264 for more details.

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/12/23 17:11:01 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [6]:
path_to_release_folder = "gs://open-targets-data-releases/25.12/"

si = StudyIndex.from_parquet(session, path_to_release_folder + "output/study/")

In [ ]:
si_g = si.df.filter(f.col("studyType") == "gwas").cache()
si_g.count()

135644

In [8]:
si_g.toPandas().to_csv("si_gwas_2512.csv", index=False)

In [ ]:
path_to_release_folder = "gs://opentargets-pipeline-runs/szsz/25.12-ppp-testrun-3/"


si = StudyIndex.from_parquet(session, path_to_release_folder + "output/study/")
sl = StudyLocus.from_parquet(session, path_to_release_folder + "output/credible_set/")

In [5]:
si.df.count()

2983597

In [6]:
sl.df.count()

4708836

In [ ]:
sl.df.filter(f.col("studyType") == "gwas").count()

1455380

In [ ]:
sl.df.filter(f.col("studyType") == "gwas").groupBy("studyId").count().orderBy(
    "count", ascending=False
).show(10)

+------------+-----+
|     studyId|count|
+------------+-----+
|GCST90691864| 4830|
|GCST90692713| 4581|
|GCST90691757| 3274|
|GCST90692661| 3128|
|GCST90429608| 3126|
|GCST90239652| 2814|
|GCST90302890| 2769|
|GCST90245848| 2685|
|GCST90475348| 2569|
|GCST90310171| 2494|
+------------+-----+
only showing top 10 rows



In [ ]:
sl.df.filter(f.col("studyType") == "gwas").join(
    si.df.select("studyId", "projectId"), on="studyId", how="inner"
).groupBy("projectId").count().show(100)

+--------------------+-------+
|           projectId|  count|
+--------------------+-------+
|         FINNGEN_R12|  20729|
|FINNGEN_R12_UKB_M...|  14790|
|                GCST|1419861|
+--------------------+-------+



In [ ]:
1400000 - 800000

600000

In [ ]:
sl_df = sl.df.filter(f.col("studyType") == "gwas").drop("locus").cache()
sl_df.count()

25/11/12 18:03:21 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


1445837

In [ ]:
sl_df = sl_df.join(
    si.df.select("studyId", "pubmedId", "diseaseIds"), on="studyId", how="inner"
).cache()
sl_df.count()

1445837

In [ ]:
sl_df.groupBy("pubmedId").count().orderBy("count", ascending=False).show(10)

+--------+------+
|pubmedId| count|
+--------+------+
|40968291|263248|
|41044249|207910|
|39024449|205406|
|40789849| 51168|
|39278973| 40560|
|39901012| 32905|
|40770095| 32902|
|34017140| 29959|
|32888494| 25110|
|32888493| 22756|
+--------+------+
only showing top 10 rows



In [ ]:
sl_df.groupBy("diseaseIds").count().orderBy("count", ascending=False).show(10)

+---------------+-----+
|     diseaseIds|count|
+---------------+-----+
|[OBA_VT0001253]|40007|
|  [EFO_0004995]|24192|
|  [EFO_0004340]|22261|
|  [OBA_0003747]|21000|
|  [OBA_0003460]|20369|
|  [EFO_0004309]|20214|
|  [EFO_0006335]|19898|
|  [EFO_0004612]|19456|
|  [EFO_0005106]|18302|
|  [EFO_0004346]|16363|
+---------------+-----+
only showing top 10 rows



25/11/12 18:40:59 WARN HeartbeatReceiver: Removing executor driver with no recent heartbeats: 979173 ms exceeds timeout 120000 ms
25/11/12 18:40:59 WARN SparkContext: Killing executors is not supported by current scheduler.
25/11/12 18:41:00 ERROR Inbox: Ignoring error
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:56)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:310)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRefByURI(RpcEnv.scala:102)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRef(RpcEnv.scala:110)
	at org.apache.spark.util.RpcUtils$.makeDriverRef(RpcUtils.scala:36)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.driverEndpoint$lzycompute(BlockManagerMasterEndpoint.scala:124)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.org$apache$spark$storage$BlockManagerMasterEndpoint$$